# Chapter 45: Survival and Time-to-Event Analysis

Synthetic NRG distributor records demonstrate censoring, Kaplan-Meier curves, horizon summaries, and restricted mean survival.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.survival import kaplan_meier,survival_at,median_survival,restricted_mean_survival
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(45);n=240
group=np.repeat(['standard','enhanced'],n//2)
scale=np.where(group=='standard',25.0,38.0)
event_time=rng.exponential(scale)
censor_time=rng.uniform(18,36,n)
duration=np.minimum(event_time,censor_time);event=(event_time<=censor_time).astype(int)
print(f'Distributors: {n}; observed churns: {event.sum()}; censored: {(1-event).sum()}')


Distributors: 240; observed churns: 136; censored: 104


In [ ]:
curves={g:kaplan_meier(duration[group==g],event[group==g]) for g in ['standard','enhanced']}
for g,c in curves.items():
 s12,s24=survival_at(c,[12,24]);med=median_survival(c)
 print(f'{g}: S(12)={s12:.3f}; S(24)={s24:.3f}; median={med:.1f}')


standard: S(12)=0.567; S(24)=0.368; median=15.4
enhanced: S(12)=0.758; S(24)=0.585; median=inf


In [ ]:
for g,c in curves.items(): print(f'{g} RMST through 24 months: {restricted_mean_survival(c,24):.2f}')


standard RMST through 24 months: 14.92
enhanced RMST through 24 months: 18.36


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for g,c in curves.items(): axes[0].step(np.r_[0,c['time']],np.r_[1,c['survival']],where='post',label=g)
axes[0].set(xlabel='Months',ylabel='Estimated survival',ylim=(0,1.02),title='Distributor retention');axes[0].legend()
h=[6,12,18,24,30];w=.36;x=np.arange(len(h))
for j,g in enumerate(['standard','enhanced']): axes[1].bar(x+(j-.5)*w,[np.sum((duration[group==g]>=t)) for t in h],w,label=g)
axes[1].set(xticks=x,xticklabels=h,xlabel='Months',ylabel='Number at risk',title='Follow-up support');axes[1].legend();fig.tight_layout();plt.show()


## Interpretation

The enhanced group has higher synthetic retention, but this observational comparison is not a causal estimate. Production work needs uncertainty, entry rules, censoring diagnostics, calendar validation, and checks for competing events and treatment selection.


In [ ]:
# Practice: choose a business horizon and compare restricted mean survival at that horizon.
